# Therapeutic Alignment Evaluation - Memory INCLUDED (Synthetic Patients)

This notebook evaluates the **original therapist responses** from 5 synthetic therapy patients,
each with **7 sessions** of therapy transcripts.

## Key Differences from Memory NOT Included Version
- **Memory Access**: YES - evaluators see BOTH conversation context AND extracted memories
- **Response Source**: Original therapist responses from transcript (NOT LLM-generated)
- **What's Evaluated**: Therapist responses from the synthetic transcripts
- **Session Processing**: Sequential with accumulating memory across sessions

## Patients
| Patient | Sessions | Output Dir |
|---------|----------|------------|
| elena_vasquez | 7 | `output_therapy_memincluded_elena_vasquez/` |
| james_o_brien | 7 | `output_therapy_memincluded_james_o_brien/` |
| marcus_williams | 7 | `output_therapy_memincluded_marcus_williams/` |
| priya_sharma | 7 | `output_therapy_memincluded_priya_sharma/` |
| sarah_chen | 7 | `output_therapy_memincluded_sarah_chen/` |

Each patient gets its own:
- Output directory with per-session checkpoints, results, images, and markdown logs
- ChromaDB vector store and collection (memory accumulates across all 7 sessions)
- Mem0 user ID

## Metrics
- **CBT Adherence Score (1-10)**: Does the therapist use CBT techniques? (evaluated WITH memory context)
- **Persona Consistency Score (1-10)**: Does the therapist maintain professional boundaries? (evaluated WITH memory context)

In [1]:
# Cell 1: Imports and Setup
import sys
import os
import json
import time
import re
import shutil
from pathlib import Path
from datetime import datetime
from dataclasses import asdict
from typing import List, Dict, Any

# Add our-pipeline to path
pipeline_path = Path("./our-pipeline")
if str(pipeline_path) not in sys.path:
    sys.path.insert(0, str(pipeline_path))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import existing modules
from transcript_parser import (
    parse_gemma_transcript_file,
    get_counselor_turns,
    get_patient_turns,
    get_conversation_context,
    ConversationTurn
)
from therapeutic_framework import (
    CBT_SYSTEM_PROMPT,
    CBT_ADHERENCE_RUBRIC,
    PERSONA_CONSISTENCY_RUBRIC,
    COGNITIVE_DISTORTIONS
)
from alignment_evaluators import (
    evaluate_cbt_adherence_with_memory,
    evaluate_persona_consistency_with_memory,
    calculate_statistics,
    calculate_decay_point
)
from mem0_integration import (
    initialize_mem0,
    create_mem0_config_with_llm,
    add_conversation_turn_to_memory,
    get_all_memories,
    get_memory_at_turn,
    search_relevant_memories,
    format_memories_for_audit,
    audit_memories
)

print("All modules loaded successfully!")

All modules loaded successfully!


In [ ]:
# Cell 2: Configuration

# ============================================================
# Lambda Cloud Configuration - IP: 129.159.33.100
# ============================================================
# IMPORTANT: Before running this notebook, start SSH tunnel:
#   ssh -L 11434:localhost:11434 ubuntu@129.159.33.100
# Keep the SSH terminal open while running the notebook.
# ============================================================

# OPTION A: Use Ollama (local, free)
USE_OLLAMA = False
OLLAMA_MODEL = "gpt-oss:20b"

# OPTION B: Use Lambda Cloud GPU instance (ENABLED)
USE_LAMBDA_CLOUD = True
LAMBDA_CLOUD_BASE_URL = "http://localhost:11434/v1"  # Via SSH tunnel to 129.159.33.100
LAMBDA_CLOUD_MODEL = "gpt-oss:20b"

# OPTION C: Use OpenAI API (requires API key)
USE_OPENAI = False
OPENAI_MODEL = "gpt-4o-mini"

# Separate model configuration for judge
if USE_LAMBDA_CLOUD:
    MODEL = LAMBDA_CLOUD_MODEL
elif USE_OLLAMA:
    MODEL = OLLAMA_MODEL
elif USE_OPENAI:
    MODEL = OPENAI_MODEL

print(f"Configuration:")
print(f"  Backend: {'Lambda Cloud (129.159.33.100)' if USE_LAMBDA_CLOUD else 'Ollama' if USE_OLLAMA else 'OpenAI'}")
print(f"  Judge Model: {MODEL}")
print(f"  Memory Access: YES (memory INCLUDED in evaluation)")
print(f"  SSH Tunnel: ssh -L 11434:localhost:11434 ubuntu@129.159.33.100")

In [ ]:
# Cell 3: Initialize OpenAI Client and Test Connection
import requests
from openai import OpenAI

if USE_LAMBDA_CLOUD:
    # Test connection to Lambda Cloud via SSH tunnel
    print("Testing connection to Lambda Cloud via SSH tunnel...")
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=10)
        if response.status_code == 200:
            models = response.json().get("models", [])
            print(f"  Connected! Available models: {[m['name'] for m in models]}")
        else:
            print(f"  Warning: Unexpected response {response.status_code}")
    except requests.exceptions.ConnectionError:
        print("  ERROR: Cannot connect to localhost:11434")
        print("  Make sure SSH tunnel is running:")
        print("    ssh -L 11434:localhost:11434 ubuntu@129.159.33.100")
        raise ConnectionError("SSH tunnel not active or Lambda Ollama not running")
    
    client = OpenAI(
        base_url=LAMBDA_CLOUD_BASE_URL,
        api_key="lambda"
    )
    print(f"\nUsing Lambda Cloud GPU instance (129.159.33.100)")
    print(f"  Base URL: {LAMBDA_CLOUD_BASE_URL}")
    print(f"  Model: {MODEL}")
elif USE_OLLAMA:
    client = OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    print(f"Using Ollama with model: {MODEL}")
elif USE_OPENAI:
    client = OpenAI()
    print(f"Using OpenAI with model: {MODEL}")
else:
    raise ValueError("Please set one of USE_OLLAMA, USE_LAMBDA_CLOUD, or USE_OPENAI to True")

print("\nClient created successfully!")

In [ ]:
# Cell 4: Define patients and parse transcripts

NOTEBOOK_TYPE = "therapy_memincluded"
NUM_SESSIONS = 7
TRANSCRIPT_DIR = Path("./output")

PATIENT_NAMES = [
    "elena_vasquez",
    "james_o_brien",
    "marcus_williams",
    "priya_sharma",
    "sarah_chen",
]

PATIENTS = []
for name in PATIENT_NAMES:
    sessions = [
        str(TRANSCRIPT_DIR / f"{name}_session{s}.txt")
        for s in range(1, NUM_SESSIONS + 1)
    ]
    PATIENTS.append({
        "id": name,
        "sessions": sessions,
        "output_dir": f"./output_{NOTEBOOK_TYPE}_{name}",
        "chroma_path": f"./chroma_db_{NOTEBOOK_TYPE}_{name}",
        "chroma_collection": f"{NOTEBOOK_TYPE}_{name}",
        "user_id": f"patient_{name}",
    })

# Parse all transcripts (per session)
for patient in PATIENTS:
    print(f"\nParsing transcripts for {patient['id']}:")
    patient["session_turns"] = []
    total_turns = 0
    total_counselor = 0
    total_patient = 0
    for si, session_file in enumerate(patient["sessions"], 1):
        turns = parse_gemma_transcript_file(session_file)
        c_turns = get_counselor_turns(turns)
        p_turns = get_patient_turns(turns)
        patient["session_turns"].append({
            "session_num": si,
            "file": session_file,
            "all_turns": turns,
            "counselor_turns": c_turns,
            "patient_turns": p_turns,
        })
        total_turns += len(turns)
        total_counselor += len(c_turns)
        total_patient += len(p_turns)
        print(f"  Session {si}: {len(turns)} turns ({len(c_turns)} counselor, {len(p_turns)} patient)")
    patient["total_turns"] = total_turns
    patient["total_counselor"] = total_counselor
    patient["total_patient"] = total_patient

print(f"\n{'='*60}")
print(f"All {len(PATIENTS)} patients ({NUM_SESSIONS} sessions each) parsed successfully!")
print(f"Total across all patients: {sum(p['total_turns'] for p in PATIENTS)} turns")

In [ ]:
# Cell 5: Helper functions for checkpoints and markdown logging

DELAY_BETWEEN_CALLS = 0.1 if (USE_OLLAMA or USE_LAMBDA_CLOUD) else 0.5
RESUME_FROM_CHECKPOINT = True
VERBOSE = True
RESET_MEMORIES = False  # Set to True for fresh run

# Memory Retrieval Strategy
USE_SEMANTIC_SEARCH = True  # True = use relevance-based retrieval, False = chronological retrieval
MEMORY_SEARCH_LIMIT = 20  # Max number of relevant memories to retrieve
MEMORY_SEARCH_THRESHOLD = None  # Optional: minimum similarity score

def get_checkpoint_path(output_dir: str, session_num: int = None) -> Path:
    if session_num is not None:
        return Path(output_dir) / f"session{session_num}" / "checkpoints" / "checkpoint.json"
    return Path(output_dir) / "checkpoints" / "checkpoint.json"

def get_markdown_path(output_dir: str, session_num: int = None) -> Path:
    if session_num is not None:
        return Path(output_dir) / f"session{session_num}" / "evaluation_log.md"
    return Path(output_dir) / "evaluation_log.md"

def load_checkpoint(output_dir: str, session_num: int = None):
    checkpoint_path = get_checkpoint_path(output_dir, session_num)
    if checkpoint_path.exists() and RESUME_FROM_CHECKPOINT:
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        print(f"  Loaded checkpoint: {checkpoint['last_turn_idx']} turns completed")
        return checkpoint
    return None

def save_checkpoint(output_dir: str, checkpoint_data: dict, session_num: int = None):
    checkpoint_path = get_checkpoint_path(output_dir, session_num)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump(checkpoint_data, f, indent=2, ensure_ascii=False)

def get_patient_turn_before(turns: list, counselor_turn_number: int):
    """Get the patient turn immediately before a counselor turn."""
    patient_turn = None
    for t in turns:
        if t.turn_number >= counselor_turn_number:
            break
        if t.role == "patient":
            patient_turn = t
    return patient_turn

def init_markdown_log(output_dir: str, patient_id: str, transcript_file: str, total_turns: int,
                      counselor_count: int, patient_count: int, session_num: int = None):
    md_path = get_markdown_path(output_dir, session_num)
    md_path.parent.mkdir(parents=True, exist_ok=True)
    session_label = f" - Session {session_num}" if session_num else ""
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(f"# Therapy Evaluation Log: {patient_id}{session_label}\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write(f"**Transcript:** {transcript_file}\n\n")
        f.write(f"**Judge Model:** {MODEL}\n\n")
        f.write(f"**Mode:** MEMORY INCLUDED (evaluators see conversation context AND memories)\n\n")
        f.write(f"**Evaluation Type:** Original therapist responses (from transcript)\n\n")
        f.write(f"**Memory Retrieval:** {'SEMANTIC SEARCH' if USE_SEMANTIC_SEARCH else 'CHRONOLOGICAL'}\n\n")
        if session_num:
            f.write(f"**Session:** {session_num} of {NUM_SESSIONS}\n\n")
        f.write(f"## Transcript Info\n\n")
        f.write(f"- Total Turns: {total_turns}\n")
        f.write(f"- Counselor Turns: {counselor_count}\n")
        f.write(f"- Patient Turns: {patient_count}\n\n")
        f.write(f"---\n\n")
        f.write(f"## Turn-by-Turn Evaluations\n\n")

def append_turn_to_markdown(output_dir: str, turn_number: int, patient_query: str, counselor_response: str,
                            cbt_score: int, cbt_reasoning: str, persona_score: int, persona_reasoning: str,
                            memory_count: int, memories_in_context: int, new_memories_this_turn: list,
                            memories_used_in_evaluation: list, session_num: int = None):
    md_path = get_markdown_path(output_dir, session_num)
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"### Turn {turn_number}\n\n")
        f.write(f"**Patient:**\n> {patient_query[:500]}{'...' if len(patient_query) > 500 else ''}\n\n")
        f.write(f"**Counselor Response:**\n> {counselor_response[:500]}{'...' if len(counselor_response) > 500 else ''}\n\n")
        f.write(f"**CBT Adherence Score:** {cbt_score}/10\n\n")
        f.write(f"**CBT Reasoning:**\n> {cbt_reasoning}\n\n")
        f.write(f"**Persona Consistency Score:** {persona_score}/10\n\n")
        f.write(f"**Persona Reasoning:**\n> {persona_reasoning}\n\n")
        f.write(f"**Memory Stats (USED in evaluation):**\n")
        f.write(f"- Total memories accumulated: {memory_count}\n")
        f.write(f"- Memories passed to evaluator: {memories_in_context}\n")
        f.write(f"- New memories this turn: {len(new_memories_this_turn)}\n\n")
        if memories_used_in_evaluation:
            f.write(f"**Memories Used in Evaluation Context:**\n")
            for i, mem in enumerate(memories_used_in_evaluation[:10], 1):
                memory_text = mem.get("memory", mem.get("text", str(mem)))
                metadata = mem.get("metadata", {})
                turn_num = metadata.get("turn_number", "?")
                role = metadata.get("role", "?")
                f.write(f"{i}. `[Turn {turn_num}, {role}]` {memory_text[:150]}{'...' if len(str(memory_text)) > 150 else ''}\n")
            if len(memories_used_in_evaluation) > 10:
                f.write(f"\n... and {len(memories_used_in_evaluation) - 10} more memories in context\n")
            f.write("\n")
        if new_memories_this_turn:
            f.write(f"**New Memories Extracted:**\n")
            for mem in new_memories_this_turn:
                mem_text = mem.get("memory", mem.get("text", str(mem)))
                metadata = mem.get("metadata", {})
                role = metadata.get("role", "?")
                turn = metadata.get("turn_number", "?")
                f.write(f"- `[Turn {turn}, {role}]` {mem_text[:200]}{'...' if len(str(mem_text)) > 200 else ''}\n")
            f.write("\n")
        f.write(f"---\n\n")

def append_memories_to_markdown(output_dir: str, memories: list, session_num: int = None):
    md_path = get_markdown_path(output_dir, session_num)
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"## Complete Memory Dump (USED in evaluation)\n\n")
        f.write(f"Total memories accumulated: {len(memories)}\n\n")
        for i, mem in enumerate(memories, 1):
            memory_text = mem.get("memory", mem.get("text", str(mem)))
            metadata = mem.get("metadata", {})
            turn = metadata.get('turn_number', '?')
            role = metadata.get('role', '?')
            f.write(f"{i}. **[Turn {turn}, {role}]** {memory_text}\n\n")

def append_summary_to_markdown(output_dir: str, cbt_results: list, persona_results: list,
                               memory_count: int, session_num: int = None):
    md_path = get_markdown_path(output_dir, session_num)
    cbt_scores = [r["score"] for r in cbt_results]
    persona_scores = [r["score"] for r in persona_results]
    with open(md_path, 'a', encoding='utf-8') as f:
        f.write(f"## Summary Statistics\n\n")
        f.write(f"### Overall CBT Adherence\n\n")
        f.write(f"- Mean: {sum(cbt_scores)/len(cbt_scores):.2f}/10\n")
        f.write(f"- Min: {min(cbt_scores)}/10\n")
        f.write(f"- Max: {max(cbt_scores)}/10\n\n")
        f.write(f"### Overall Persona Consistency\n\n")
        f.write(f"- Mean: {sum(persona_scores)/len(persona_scores):.2f}/10\n")
        f.write(f"- Min: {min(persona_scores)}/10\n")
        f.write(f"- Max: {max(persona_scores)}/10\n\n")
        f.write(f"### Memory (USED in evaluation)\n\n")
        f.write(f"- Total Memories Stored: {memory_count}\n\n")

def truncate(text: str, length: int = 80) -> str:
    return text[:length] + "..." if len(text) > length else text

print("Helper functions defined.")

In [ ]:
# Cell 6: Main Processing Loop - Iterates over all patients and sessions
# Evaluates ORIGINAL therapist responses from transcripts (not LLM-generated)
# WITH memory context passed to evaluators
# Memory accumulates across sessions (session1 -> session7)

all_patient_results = {}

for patient in PATIENTS:
    patient_id = patient["id"]
    output_dir = Path(patient["output_dir"])
    user_id = patient["user_id"]
    chroma_path = patient["chroma_path"]
    chroma_collection = patient["chroma_collection"]

    print(f"\n{'#'*60}")
    print(f"# PROCESSING PATIENT: {patient_id}")
    print(f"# Sessions: {NUM_SESSIONS}")
    print(f"# Output: {output_dir}")
    print(f"# ChromaDB: {chroma_path} / {chroma_collection}")
    print(f"# MODE: Memory INCLUDED (sequential sessions, accumulating memory)")
    print(f"{'#'*60}")

    # Create output directories
    output_dir.mkdir(exist_ok=True)
    (output_dir / "images").mkdir(exist_ok=True)
    for si in range(1, NUM_SESSIONS + 1):
        session_dir = output_dir / f"session{si}"
        session_dir.mkdir(exist_ok=True)
        (session_dir / "checkpoints").mkdir(exist_ok=True)
        (session_dir / "results").mkdir(exist_ok=True)

    # Initialize Mem0 for this patient (persists across all sessions)
    if RESET_MEMORIES and Path(chroma_path).exists():
        shutil.rmtree(chroma_path)
        print(f"Deleted existing {chroma_path} folder for fresh start")

    if USE_LAMBDA_CLOUD:
        mem_config = create_mem0_config_with_llm(
            llm_provider="ollama",
            model=LAMBDA_CLOUD_MODEL,
            base_url=LAMBDA_CLOUD_BASE_URL.replace("/v1", "")
        )
    elif USE_OLLAMA:
        mem_config = create_mem0_config_with_llm(
            llm_provider="ollama",
            model=OLLAMA_MODEL,
            base_url="http://localhost:11434"
        )
    elif USE_OPENAI:
        mem_config = None

    if mem_config:
        mem_config["vector_store"]["config"]["collection_name"] = chroma_collection
        mem_config["vector_store"]["config"]["path"] = chroma_path
        memory = initialize_mem0(config=mem_config, reset_collection=RESET_MEMORIES)
    else:
        from mem0 import Memory
        memory = Memory()

    print(f"Mem0 initialized for {patient_id} (memories WILL be used in evaluation)")
    print(f"  Collection: {chroma_collection}")
    print(f"  Path: {chroma_path}")

    # Track results across all sessions for this patient
    all_cbt_results = []
    all_persona_results = []
    all_memory_snapshots = []
    session_summaries = []
    global_turn_counter = 0  # Track cumulative turn number across sessions

    # Track previous memories for detecting new ones
    previous_memory_ids = set()
    initial_memories = get_all_memories(memory, user_id)
    for mem in initial_memories:
        previous_memory_ids.add(mem.get("id", str(mem)))

    # Process each session sequentially (memory accumulates)
    for session_data in patient["session_turns"]:
        session_num = session_data["session_num"]
        session_file = session_data["file"]
        all_turns = session_data["all_turns"]
        counselor_turns = session_data["counselor_turns"]
        patient_turns_list = session_data["patient_turns"]

        print(f"\n  {'='*50}")
        print(f"  SESSION {session_num}/{NUM_SESSIONS}: {session_file}")
        print(f"  Turns: {len(all_turns)} | Counselor: {len(counselor_turns)} | Patient: {len(patient_turns_list)}")
        print(f"  Accumulated memories so far: {len(get_all_memories(memory, user_id))}")
        print(f"  {'='*50}")

        # Load session checkpoint if exists
        checkpoint = load_checkpoint(str(output_dir), session_num)

        if checkpoint:
            session_cbt_results = checkpoint.get('cbt_results', [])
            session_persona_results = checkpoint.get('persona_results', [])
            session_memory_snapshots = checkpoint.get('memory_snapshots', [])
            last_turn_idx = checkpoint.get('last_turn_idx', 0)
        else:
            session_cbt_results = []
            session_persona_results = []
            session_memory_snapshots = []
            last_turn_idx = 0
            init_markdown_log(
                str(output_dir), patient_id, session_file,
                len(all_turns), len(counselor_turns), len(patient_turns_list),
                session_num=session_num
            )

        # Store baseline for persona comparison (first counselor response of session 1)
        if session_num == 1 and counselor_turns:
            baseline_response = counselor_turns[0].content
        elif not hasattr(patient, '_baseline') and counselor_turns:
            baseline_response = counselor_turns[0].content

        counselor_turn_numbers = {t.turn_number for t in counselor_turns}
        remaining_turns = all_turns[last_turn_idx:]

        for i, turn in enumerate(remaining_turns):
            current_idx = last_turn_idx + i
            global_turn = global_turn_counter + turn.turn_number

            if VERBOSE:
                role_label = "COUNSELOR" if turn.role == "counselor" else "PATIENT"
                print(f"\n    [S{session_num} T{turn.turn_number}] {role_label}: {truncate(turn.content, 70)}")

            # 1. Add turn to mem0 (memory persists across sessions)
            add_conversation_turn_to_memory(
                memory=memory,
                turn_content=turn.content,
                role=turn.role,
                turn_number=global_turn,
                user_id=user_id,
                verbose=VERBOSE
            )

            # 2. Only evaluate COUNSELOR turns
            if turn.role == "counselor" and turn.turn_number in counselor_turn_numbers:
                patient_turn_before = get_patient_turn_before(all_turns, turn.turn_number)
                patient_query = patient_turn_before.content if patient_turn_before else "(No preceding patient turn)"

                # Get memories
                if USE_SEMANTIC_SEARCH:
                    search_query = f"{patient_query} {turn.content}"
                    memories_up_to_turn = search_relevant_memories(
                        memory=memory, query=search_query, user_id=user_id,
                        limit=MEMORY_SEARCH_LIMIT, threshold=MEMORY_SEARCH_THRESHOLD, rerank=True
                    )
                else:
                    memories_up_to_turn = get_memory_at_turn(
                        memory=memory, turn_number=global_turn, user_id=user_id
                    )

                memories_formatted = format_memories_for_audit(memories_up_to_turn)

                # Evaluate CBT adherence WITH memory context
                cbt_result = evaluate_cbt_adherence_with_memory(
                    client=client, counselor_response=turn.content,
                    conversation_context="", memories_context=memories_formatted,
                    turn_number=global_turn, model=MODEL
                )
                cbt_result_dict = asdict(cbt_result)
                cbt_result_dict["session"] = session_num
                cbt_result_dict["global_turn"] = global_turn
                session_cbt_results.append(cbt_result_dict)

                time.sleep(DELAY_BETWEEN_CALLS)

                # Evaluate persona consistency WITH memory context
                persona_result = evaluate_persona_consistency_with_memory(
                    client=client, counselor_response=turn.content,
                    baseline_response=baseline_response, conversation_context="",
                    memories_context=memories_formatted, turn_number=global_turn, model=MODEL
                )
                persona_result_dict = asdict(persona_result)
                persona_result_dict["session"] = session_num
                persona_result_dict["global_turn"] = global_turn
                session_persona_results.append(persona_result_dict)

                # Track new memories
                current_memories = get_all_memories(memory, user_id)
                current_memory_ids = {mem.get("id", str(mem)) for mem in current_memories}
                new_memory_ids = current_memory_ids - previous_memory_ids
                new_memories_this_turn = [mem for mem in current_memories if mem.get("id", str(mem)) in new_memory_ids]
                previous_memory_ids = current_memory_ids

                session_memory_snapshots.append({
                    "turn_number": turn.turn_number,
                    "global_turn": global_turn,
                    "session": session_num,
                    "memory_count": len(current_memories),
                    "memories_in_context": len(memories_up_to_turn),
                    "new_memories_this_turn": len(new_memories_this_turn),
                    "cbt_score": cbt_result.score,
                    "persona_score": persona_result.score
                })

                if VERBOSE:
                    print(f"      --> EVALUATED: CBT: {cbt_result.score}/10 | Persona: {persona_result.score}/10")
                    print(f"      --> Memories in context: {len(memories_up_to_turn)} | Total: {len(current_memories)}")

                append_turn_to_markdown(
                    str(output_dir), turn_number=turn.turn_number,
                    patient_query=patient_query, counselor_response=turn.content,
                    cbt_score=cbt_result.score, cbt_reasoning=cbt_result.reasoning,
                    persona_score=persona_result.score, persona_reasoning=persona_result.reasoning,
                    memory_count=len(current_memories), memories_in_context=len(memories_up_to_turn),
                    new_memories_this_turn=new_memories_this_turn,
                    memories_used_in_evaluation=memories_up_to_turn, session_num=session_num
                )

                time.sleep(DELAY_BETWEEN_CALLS)

            # Save checkpoint after each turn
            checkpoint_data = {
                'last_turn_idx': current_idx + 1,
                'total_turns': len(all_turns),
                'session': session_num,
                'cbt_results': session_cbt_results,
                'persona_results': session_persona_results,
                'memory_snapshots': session_memory_snapshots,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }
            save_checkpoint(str(output_dir), checkpoint_data, session_num)

        # Session complete - save session results
        session_memories = get_all_memories(memory, user_id)
        append_memories_to_markdown(str(output_dir), session_memories, session_num)
        if session_cbt_results:
            append_summary_to_markdown(str(output_dir), session_cbt_results, session_persona_results,
                                       len(session_memories), session_num)

        # Save session results JSON
        session_results_path = output_dir / f"session{session_num}" / "results" / "results.json"
        session_results_data = {
            "patient_id": patient_id,
            "session": session_num,
            "transcript_file": session_file,
            "total_turns": len(all_turns),
            "counselor_turns_evaluated": len(session_cbt_results),
            "judge_model": MODEL,
            "memory_enhanced": True,
            "evaluation_mode": "memory_included",
            "cbt_adherence_results": session_cbt_results,
            "persona_consistency_results": session_persona_results,
            "memory_snapshots": session_memory_snapshots
        }
        with open(session_results_path, "w", encoding="utf-8") as f:
            json.dump(session_results_data, f, indent=2, ensure_ascii=False)

        # Accumulate for cross-session
        avg_cbt = sum(r['score'] for r in session_cbt_results) / len(session_cbt_results) if session_cbt_results else 0
        avg_persona = sum(r['score'] for r in session_persona_results) / len(session_persona_results) if session_persona_results else 0
        session_summaries.append({
            "session": session_num,
            "avg_cbt": avg_cbt,
            "avg_persona": avg_persona,
            "memory_count": len(session_memories),
            "turns_evaluated": len(session_cbt_results),
        })
        all_cbt_results.extend(session_cbt_results)
        all_persona_results.extend(session_persona_results)
        all_memory_snapshots.extend(session_memory_snapshots)

        # Update global turn counter
        global_turn_counter += len(all_turns)

        print(f"\n  Session {session_num} COMPLETE: CBT avg={avg_cbt:.2f}, Persona avg={avg_persona:.2f}, Memories={len(session_memories)}")

    # Patient complete - save combined results
    final_memories = get_all_memories(memory, user_id)

    all_patient_results[patient_id] = {
        "cbt_results": all_cbt_results,
        "persona_results": all_persona_results,
        "memory_snapshots": all_memory_snapshots,
        "session_summaries": session_summaries,
        "final_memory_count": len(final_memories),
        "output_dir": str(output_dir),
    }

    print(f"\n{'='*60}")
    print(f"PATIENT {patient_id} COMPLETE! ({NUM_SESSIONS} sessions)")
    print(f"  Total counselor turns evaluated: {len(all_cbt_results)}")
    print(f"  Total memories: {len(final_memories)}")
    if all_cbt_results:
        avg_cbt = sum(r['score'] for r in all_cbt_results) / len(all_cbt_results)
        avg_persona = sum(r['score'] for r in all_persona_results) / len(all_persona_results)
        print(f"  Overall Avg CBT Score: {avg_cbt:.2f}/10")
        print(f"  Overall Avg Persona Score: {avg_persona:.2f}/10")
    print(f"{'='*60}")

print(f"\n\n{'#'*60}")
print(f"ALL {len(PATIENTS)} PATIENTS PROCESSED! ({NUM_SESSIONS} sessions each)")
print(f"{'#'*60}")

In [ ]:
# Cell 7: Memory Audit for each patient

for patient in PATIENTS:
    patient_id = patient["id"]
    user_id = patient["user_id"]
    chroma_path = patient["chroma_path"]
    chroma_collection = patient["chroma_collection"]
    output_dir = Path(patient["output_dir"])

    print(f"\n{'='*60}")
    print(f"Memory Audit for {patient_id} (memories WERE used in evaluation)")
    print(f"{'='*60}")

    # Re-initialize mem0 to access stored memories
    if USE_LAMBDA_CLOUD:
        mem_config = create_mem0_config_with_llm(
            llm_provider="ollama", model=LAMBDA_CLOUD_MODEL,
            base_url=LAMBDA_CLOUD_BASE_URL.replace("/v1", "")
        )
    elif USE_OLLAMA:
        mem_config = create_mem0_config_with_llm(
            llm_provider="ollama", model=OLLAMA_MODEL, base_url="http://localhost:11434"
        )
    elif USE_OPENAI:
        mem_config = None

    if mem_config:
        mem_config["vector_store"]["config"]["collection_name"] = chroma_collection
        mem_config["vector_store"]["config"]["path"] = chroma_path
        memory = initialize_mem0(config=mem_config, reset_collection=False)
    else:
        from mem0 import Memory
        memory = Memory()

    all_memories = get_all_memories(memory, user_id)
    print(f"Total memories stored (across {NUM_SESSIONS} sessions): {len(all_memories)}")

    audit_result = audit_memories(client=client, memories=all_memories, model=MODEL)

    print(f"\nMemory Audit Results:")
    print(f"  Total Memories: {audit_result.total_memories}")
    print(f"  Distortion Count: {audit_result.distortion_count}")
    print(f"  Collusion Score: {audit_result.collusion_score:.2f}")
    print(f"  Reasoning: {audit_result.reasoning}")

    all_patient_results[patient_id]["memory_audit"] = asdict(audit_result)

In [ ]:
# Cell 8: Save final combined results for each patient

for patient in PATIENTS:
    patient_id = patient["id"]
    result_data = all_patient_results[patient_id]
    output_dir = Path(patient["output_dir"])

    cbt_results = result_data["cbt_results"]
    persona_results = result_data["persona_results"]
    audit = result_data.get("memory_audit", {})

    output = {
        "metadata": {
            "patient_id": patient_id,
            "notebook_type": NOTEBOOK_TYPE,
            "num_sessions": NUM_SESSIONS,
            "total_counselor_turns": patient["total_counselor"],
            "judge_model": MODEL,
            "mode": "memory_included",
            "evaluation_type": "original_therapist_responses",
            "evaluator_receives": "conversation_context_and_memories",
            "memory_retrieval": "semantic_search" if USE_SEMANTIC_SEARCH else "chronological"
        },
        "session_summaries": result_data["session_summaries"],
        "cbt_results": cbt_results,
        "persona_results": persona_results,
        "memory_audit": audit,
        "statistics": {
            "avg_cbt_score": sum(r["score"] for r in cbt_results) / len(cbt_results) if cbt_results else 0,
            "avg_persona_score": sum(r["score"] for r in persona_results) / len(persona_results) if persona_results else 0,
            "collusion_score": audit.get("collusion_score", 0)
        }
    }

    output_file = output_dir / f"{NOTEBOOK_TYPE}_{patient_id}.json"
    with open(output_file, "w") as f:
        json.dump(output, f, indent=2)

    print(f"\n{'='*60}")
    print(f"Results for {patient_id} saved to {output_file}")
    print(f"  Sessions: {NUM_SESSIONS}")
    print(f"  Average CBT Score: {output['statistics']['avg_cbt_score']:.2f}/10")
    print(f"  Average Persona Score: {output['statistics']['avg_persona_score']:.2f}/10")
    print(f"  Memory Collusion Score: {output['statistics']['collusion_score']:.2f}")
    print(f"  Session breakdown:")
    for ss in result_data["session_summaries"]:
        print(f"    Session {ss['session']}: CBT={ss['avg_cbt']:.2f}, Persona={ss['avg_persona']:.2f}, Mems={ss['memory_count']}")

In [ ]:
# Cell 9: Visualization - Per-patient cross-session plots
import matplotlib.pyplot as plt
import numpy as np

for patient in PATIENTS:
    patient_id = patient["id"]
    result_data = all_patient_results[patient_id]
    output_dir = Path(patient["output_dir"])

    session_summaries = result_data["session_summaries"]
    memory_snapshots = result_data["memory_snapshots"]

    if not session_summaries:
        print(f"No results for {patient_id}, skipping visualization.")
        continue

    print(f"\nVisualizing {patient_id} ({len(session_summaries)} sessions)")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Plot 1: CBT & Persona scores across sessions (bar chart)
    ax1 = axes[0, 0]
    sessions = [s["session"] for s in session_summaries]
    cbt_avgs = [s["avg_cbt"] for s in session_summaries]
    persona_avgs = [s["avg_persona"] for s in session_summaries]
    x = np.arange(len(sessions))
    width = 0.35
    ax1.bar(x - width/2, cbt_avgs, width, color='steelblue', alpha=0.8, label='CBT Adherence')
    ax1.bar(x + width/2, persona_avgs, width, color='forestgreen', alpha=0.8, label='Persona Consistency')
    ax1.axhline(y=7, color='orange', linestyle='--', alpha=0.7, label='Good (7)')
    ax1.axhline(y=5, color='red', linestyle='--', alpha=0.7, label='Warning (5)')
    ax1.set_xlabel('Session')
    ax1.set_ylabel('Average Score (1-10)')
    ax1.set_title(f'{patient_id} - Scores Across Sessions')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'S{s}' for s in sessions])
    ax1.set_ylim(0, 10.5)
    ax1.legend(loc='lower left', fontsize=8)
    ax1.grid(True, alpha=0.3, axis='y')

    # Plot 2: Memory accumulation across sessions
    ax2 = axes[0, 1]
    mem_counts = [s["memory_count"] for s in session_summaries]
    ax2.bar(x, mem_counts, color='purple', alpha=0.8)
    ax2.plot(x, mem_counts, 'mo-', linewidth=2, markersize=8)
    ax2.set_xlabel('Session')
    ax2.set_ylabel('Total Accumulated Memories')
    ax2.set_title(f'{patient_id} - Memory Growth Across Sessions')
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'S{s}' for s in sessions])
    ax2.grid(True, alpha=0.3, axis='y')

    # Plot 3: Turn-level CBT scores across all sessions (continuous line)
    ax3 = axes[1, 0]
    all_cbt = result_data["cbt_results"]
    if all_cbt:
        global_turns = [r.get("global_turn", r["turn_number"]) for r in all_cbt]
        cbt_scores = [r["score"] for r in all_cbt]
        session_labels = [r.get("session", 1) for r in all_cbt]

        # Color by session
        colors = plt.cm.viridis(np.linspace(0, 1, NUM_SESSIONS))
        prev_session = None
        for j, (gt, score, sess) in enumerate(zip(global_turns, cbt_scores, session_labels)):
            color = colors[sess - 1]
            if sess != prev_session:
                ax3.axvline(x=gt, color='gray', linestyle=':', alpha=0.3)
                ax3.text(gt, 10.5, f'S{sess}', fontsize=7, ha='center', alpha=0.7)
                prev_session = sess
            ax3.scatter(gt, score, c=[color], s=15, alpha=0.6)
        ax3.axhline(y=7, color='orange', linestyle='--', alpha=0.5)
        window = max(5, len(cbt_scores) // 10)
        if len(cbt_scores) >= window:
            rolling = np.convolve(cbt_scores, np.ones(window)/window, mode='valid')
            ax3.plot(global_turns[window//2:len(rolling)+window//2], rolling, 'b-', linewidth=2, label=f'Rolling Avg ({window})')
        ax3.set_xlabel('Global Turn Number')
        ax3.set_ylabel('CBT Score')
        ax3.set_title(f'{patient_id} - CBT Adherence Over All Sessions')
        ax3.set_ylim(0, 11)
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3)

    # Plot 4: Turn-level Persona scores across all sessions
    ax4 = axes[1, 1]
    all_persona = result_data["persona_results"]
    if all_persona:
        global_turns_p = [r.get("global_turn", r["turn_number"]) for r in all_persona]
        persona_scores = [r["score"] for r in all_persona]
        session_labels_p = [r.get("session", 1) for r in all_persona]

        prev_session = None
        for j, (gt, score, sess) in enumerate(zip(global_turns_p, persona_scores, session_labels_p)):
            color = colors[sess - 1]
            if sess != prev_session:
                ax4.axvline(x=gt, color='gray', linestyle=':', alpha=0.3)
                ax4.text(gt, 10.5, f'S{sess}', fontsize=7, ha='center', alpha=0.7)
                prev_session = sess
            ax4.scatter(gt, score, c=[color], s=15, alpha=0.6)
        ax4.axhline(y=7, color='orange', linestyle='--', alpha=0.5)
        if len(persona_scores) >= window:
            rolling_p = np.convolve(persona_scores, np.ones(window)/window, mode='valid')
            ax4.plot(global_turns_p[window//2:len(rolling_p)+window//2], rolling_p, 'g-', linewidth=2, label=f'Rolling Avg ({window})')
        ax4.set_xlabel('Global Turn Number')
        ax4.set_ylabel('Persona Score')
        ax4.set_title(f'{patient_id} - Persona Consistency Over All Sessions')
        ax4.set_ylim(0, 11)
        ax4.legend(fontsize=8)
        ax4.grid(True, alpha=0.3)

    plt.suptitle(f'{patient_id} - {NOTEBOOK_TYPE} (Memory INCLUDED)', fontsize=14, y=1.02)
    plt.tight_layout()
    image_path = output_dir / "images" / "alignment_overview.png"
    plt.savefig(image_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved to {image_path}")

In [ ]:
# Cell 10: Cross-patient comparison
import matplotlib.pyplot as plt
import numpy as np

patient_ids = [p["id"] for p in PATIENTS]
cbt_means = []
persona_means = []
memory_counts = []

for pid in patient_ids:
    r = all_patient_results[pid]
    cbt_scores = [x["score"] for x in r["cbt_results"]]
    persona_scores = [x["score"] for x in r["persona_results"]]
    cbt_means.append(sum(cbt_scores)/len(cbt_scores) if cbt_scores else 0)
    persona_means.append(sum(persona_scores)/len(persona_scores) if persona_scores else 0)
    memory_counts.append(r["final_memory_count"])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.arange(len(patient_ids))
width = 0.35
short_names = [pid.replace('_', '\n') for pid in patient_ids]

# CBT comparison
ax1 = axes[0]
ax1.bar(x, cbt_means, width, color='steelblue', alpha=0.8)
ax1.axhline(y=7, color='orange', linestyle='--', label='Good (7)')
ax1.axhline(y=5, color='red', linestyle='--', label='Warning (5)')
ax1.set_xlabel('Patient')
ax1.set_ylabel('Mean CBT Score')
ax1.set_title('CBT Adherence by Patient')
ax1.set_xticks(x)
ax1.set_xticklabels(short_names, fontsize=8)
ax1.set_ylim(0, 10)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Persona comparison
ax2 = axes[1]
ax2.bar(x, persona_means, width, color='forestgreen', alpha=0.8)
ax2.axhline(y=7, color='orange', linestyle='--', label='Good (7)')
ax2.axhline(y=5, color='red', linestyle='--', label='Warning (5)')
ax2.set_xlabel('Patient')
ax2.set_ylabel('Mean Persona Score')
ax2.set_title('Persona Consistency by Patient')
ax2.set_xticks(x)
ax2.set_xticklabels(short_names, fontsize=8)
ax2.set_ylim(0, 10)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Memory count comparison
ax3 = axes[2]
ax3.bar(x, memory_counts, width, color='purple', alpha=0.8)
ax3.set_xlabel('Patient')
ax3.set_ylabel('Total Memories (USED)')
ax3.set_title(f'Memories Accumulated ({NUM_SESSIONS} sessions)')
ax3.set_xticks(x)
ax3.set_xticklabels(short_names, fontsize=8)
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Cross-Patient Comparison - {NOTEBOOK_TYPE} ({NUM_SESSIONS} sessions each)', fontsize=14, y=1.02)
plt.tight_layout()

# Save to each patient's output dir
for patient in PATIENTS:
    img_path = Path(patient["output_dir"]) / "images" / "cross_patient_comparison.png"
    plt.savefig(img_path, dpi=150, bbox_inches='tight')

plt.show()

# Print summary table
print("\n" + "=" * 80)
print(f"CROSS-PATIENT SUMMARY ({NOTEBOOK_TYPE} - {NUM_SESSIONS} sessions each)")
print("=" * 80)
print(f"{'Patient':<20} {'CBT Mean':<12} {'Persona Mean':<14} {'Memories':<10} {'Sessions':<10}")
print("-" * 80)
for i, pid in enumerate(patient_ids):
    print(f"{pid:<20} {cbt_means[i]:<12.2f} {persona_means[i]:<14.2f} {memory_counts[i]:<10} {NUM_SESSIONS:<10}")

# Session-level heatmap
print(f"\n{'='*80}")
print("SESSION-LEVEL BREAKDOWN (CBT / Persona)")
print("=" * 80)
header = f"{'Patient':<20}" + "".join(f"{'S'+str(s):<12}" for s in range(1, NUM_SESSIONS + 1))
print(header)
print("-" * 80)
for pid in patient_ids:
    r = all_patient_results[pid]
    row = f"{pid:<20}"
    for ss in r["session_summaries"]:
        row += f"{ss['avg_cbt']:.1f}/{ss['avg_persona']:.1f}    "
    print(row)